# 01 - Prophet Model (OPTIMIZED)

Ce notebook entraine un modele Prophet **optimise** pour la prevision des ventes.

**AMELIORATIONS:**
- Optimisation des hyperparametres avec Optuna
- Saisonnalites multiples (hebdo, mensuelle, annuelle)
- Regresseurs externes si disponibles
- Cross-validation temporelle robuste

**Outputs:**
- `models/artifacts/prophet_model_YYYYMMDD.pkl`
- `models/artifacts/prophet_config_YYYYMMDD.json`
- `models/metrics/prophet_metrics_YYYYMMDD.json`

In [ ]:
import sys
import json
import datetime
import warnings
from pathlib import Path
from math import sqrt

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# Setup project path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
HORIZON = 14
DATE_COL = 'date'
TARGET_COL = 'value'
TODAY = datetime.datetime.now().strftime("%Y%m%d")

# Optimisation avec Optuna
USE_OPTUNA = True
OPTUNA_TRIALS = 50  # Plus de trials pour meilleure optimisation

# Paths
ARTIFACTS_PATH = PROJECT_ROOT / "models" / "artifacts"
METRICS_PATH = PROJECT_ROOT / "models" / "metrics"
ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
METRICS_PATH.mkdir(parents=True, exist_ok=True)

# Import project config
try:
    from src.config import TRAIN_CSV, GROUP_COLS
except ImportError:
    TRAIN_CSV = PROJECT_ROOT / "data" / "interim" / "train.csv"
    GROUP_COLS = ['store_id', 'product_id']

try:
    from src.data.clean import clean_dataframe
    CLEAN_AVAILABLE = True
except ImportError:
    CLEAN_AVAILABLE = False

print(f"HORIZON={HORIZON}, TODAY={TODAY}")
print(f"USE_OPTUNA={USE_OPTUNA}, OPTUNA_TRIALS={OPTUNA_TRIALS}")

In [ ]:
# Find data file
def find_data_file():
    candidates = [
        TRAIN_CSV,
        PROJECT_ROOT / "data" / "processed" / "uploaded_generated_training_10950_features.csv",
        PROJECT_ROOT / "data" / "processed" / "train_features.csv",
    ]
    for folder in ["interim", "processed", "raw"]:
        folder_path = PROJECT_ROOT / "data" / folder
        if folder_path.exists():
            for f in folder_path.glob("*.csv"):
                candidates.append(f)
    
    for p in candidates:
        if p and p.exists():
            return p
    raise FileNotFoundError("No data file found")

DATA_PATH = find_data_file()
print(f"Data: {DATA_PATH}")

## 1. Chargement et Preparation des Donnees

In [ ]:
# Load data
df = pd.read_csv(DATA_PATH, parse_dates=[DATE_COL])
print(f"Shape: {df.shape}")

# Clean
if CLEAN_AVAILABLE:
    df = clean_dataframe(df, date_col=DATE_COL, value_col=TARGET_COL, 
                         fill_strategy='ffill', outlier_threshold=None)

# Remap columns
if 'store_id' not in df.columns and 'id' in df.columns:
    df = df.rename(columns={'id': 'store_id'})

# Handle on_promo
if 'on_promo' in df.columns:
    df['on_promo'] = df['on_promo'].map(
        {True: 1, False: 0, 'True': 1, 'False': 0, 1: 1, 0: 0}
    ).fillna(0).astype(int)

# Detect group columns
available_groups = [c for c in GROUP_COLS if c in df.columns]
print(f"Group columns: {available_groups}")

# Ensure numeric target
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors='coerce')
df = df.dropna(subset=[TARGET_COL])

# Sort by date
df = df.sort_values(DATE_COL).reset_index(drop=True)

print(f"Clean shape: {df.shape}")
print(f"Target: mean={df[TARGET_COL].mean():.2f}, std={df[TARGET_COL].std():.2f}")
print(f"Date range: {df[DATE_COL].min()} to {df[DATE_COL].max()}")

In [ ]:
# ============================================================
# PREPARATION: Agreger les donnees par date pour Prophet
# Prophet travaille avec UNE serie temporelle (ds, y)
# ============================================================

# Agregation par date
agg_dict = {TARGET_COL: 'sum'}

# Ajouter des regresseurs si disponibles
REGRESSORS = []
if 'price' in df.columns:
    agg_dict['price'] = 'mean'
    REGRESSORS.append('price')
if 'on_promo' in df.columns:
    agg_dict['on_promo'] = 'mean'
    REGRESSORS.append('on_promo')

df_agg = df.groupby(DATE_COL).agg(agg_dict).reset_index()
df_agg = df_agg.rename(columns={DATE_COL: 'ds', TARGET_COL: 'y'})

# Ajouter des features calendaires
df_agg['is_weekend'] = (df_agg['ds'].dt.dayofweek >= 5).astype(float)
REGRESSORS.append('is_weekend')

print(f"Aggregated shape: {df_agg.shape}")
print(f"Target range: [{df_agg['y'].min():.2f}, {df_agg['y'].max():.2f}]")
print(f"Regressors: {REGRESSORS}")
df_agg.head()

## 2. Train/Test Split Temporel

In [ ]:
# Temporal split
n_dates = len(df_agg)
n_test = max(HORIZON, int(n_dates * 0.15))
cutoff_idx = n_dates - n_test

df_train = df_agg.iloc[:cutoff_idx].copy()
df_test = df_agg.iloc[cutoff_idx:].copy()

print(f"Total dates: {n_dates}")
print(f"Train: {len(df_train)} samples")
print(f"Test: {len(df_test)} samples")
print(f"Cutoff date: {df_train['ds'].max()}")

## 3. Optimisation des Hyperparametres avec Optuna

In [ ]:
from prophet import Prophet

def create_prophet_model(params, regressors):
    """Create Prophet model with given parameters."""
    model = Prophet(
        growth=params.get('growth', 'linear'),
        seasonality_mode=params.get('seasonality_mode', 'multiplicative'),
        changepoint_prior_scale=params.get('changepoint_prior_scale', 0.05),
        seasonality_prior_scale=params.get('seasonality_prior_scale', 10.0),
        holidays_prior_scale=params.get('holidays_prior_scale', 10.0),
        yearly_seasonality=params.get('yearly_seasonality', True),
        weekly_seasonality=params.get('weekly_seasonality', True),
        daily_seasonality=False,
        changepoint_range=params.get('changepoint_range', 0.8),
        n_changepoints=params.get('n_changepoints', 25),
    )
    
    # Add custom seasonalities if enabled
    if params.get('add_monthly', False):
        model.add_seasonality(
            name='monthly', 
            period=30.5, 
            fourier_order=params.get('monthly_fourier', 5)
        )
    
    if params.get('add_quarterly', False):
        model.add_seasonality(
            name='quarterly', 
            period=91.25, 
            fourier_order=params.get('quarterly_fourier', 3)
        )
    
    # Add regressors
    for reg in regressors:
        model.add_regressor(reg, mode=params.get('regressor_mode', 'additive'))
    
    return model


def train_and_evaluate(df_train, df_test, params, regressors):
    """Train Prophet and return test metrics."""
    # Prepare columns
    cols = ['ds', 'y'] + regressors
    train_data = df_train[cols].copy()
    test_data = df_test[cols].copy()
    
    # Create and fit model
    model = create_prophet_model(params, regressors)
    model.fit(train_data)
    
    # Predict
    future = test_data[['ds'] + regressors].copy()
    forecast = model.predict(future)
    
    # Get predictions
    y_true = test_data['y'].values
    y_pred = forecast['yhat'].values
    
    # Clip negative predictions to 0
    y_pred = np.clip(y_pred, 0, None)
    
    # Calculate metrics
    mae = mean_absolute_error(y_true, y_pred)
    rmse = sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    return {
        'model': model,
        'forecast': forecast,
        'y_true': y_true,
        'y_pred': y_pred,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
    }

In [ ]:
if USE_OPTUNA:
    import optuna
    from optuna.samplers import TPESampler
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    
    def objective(trial):
        """Optuna objective function for Prophet."""
        params = {
            'growth': trial.suggest_categorical('growth', ['linear', 'flat']),
            'seasonality_mode': trial.suggest_categorical('seasonality_mode', ['additive', 'multiplicative']),
            'changepoint_prior_scale': trial.suggest_float('changepoint_prior_scale', 0.001, 0.5, log=True),
            'seasonality_prior_scale': trial.suggest_float('seasonality_prior_scale', 0.1, 20.0, log=True),
            'holidays_prior_scale': trial.suggest_float('holidays_prior_scale', 0.1, 20.0, log=True),
            'changepoint_range': trial.suggest_float('changepoint_range', 0.7, 0.95),
            'n_changepoints': trial.suggest_int('n_changepoints', 10, 50),
            'yearly_seasonality': trial.suggest_categorical('yearly_seasonality', [True, False]),
            'weekly_seasonality': trial.suggest_categorical('weekly_seasonality', [True, False]),
            'add_monthly': trial.suggest_categorical('add_monthly', [True, False]),
            'monthly_fourier': trial.suggest_int('monthly_fourier', 3, 10),
            'add_quarterly': trial.suggest_categorical('add_quarterly', [True, False]),
            'quarterly_fourier': trial.suggest_int('quarterly_fourier', 2, 7),
            'regressor_mode': trial.suggest_categorical('regressor_mode', ['additive', 'multiplicative']),
        }
        
        try:
            result = train_and_evaluate(df_train, df_test, params, REGRESSORS)
            # Penaliser les R2 negatifs
            if result['R2'] < 0:
                return float('inf')
            return result['RMSE']
        except Exception as e:
            return float('inf')
    
    # Run optimization
    print(f"\nStarting Optuna optimization with {OPTUNA_TRIALS} trials...")
    study = optuna.create_study(
        direction='minimize',
        sampler=TPESampler(seed=42)
    )
    study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
    
    print(f"\nBest trial:")
    print(f"  RMSE: {study.best_value:.4f}")
    print(f"  Params: {study.best_params}")
    
    best_params = study.best_params
else:
    # Default parameters
    best_params = {
        'growth': 'linear',
        'seasonality_mode': 'multiplicative',
        'changepoint_prior_scale': 0.1,
        'seasonality_prior_scale': 5.0,
        'holidays_prior_scale': 5.0,
        'changepoint_range': 0.85,
        'n_changepoints': 30,
        'yearly_seasonality': True,
        'weekly_seasonality': True,
        'add_monthly': True,
        'monthly_fourier': 5,
        'add_quarterly': False,
        'quarterly_fourier': 3,
        'regressor_mode': 'additive',
    }
    print("Using default parameters")

## 4. Entrainement du Modele Final

In [ ]:
# Train final model with best parameters
print("\nTraining final Prophet model with optimized parameters...")
result = train_and_evaluate(df_train, df_test, best_params, REGRESSORS)

model = result['model']
forecast = result['forecast']
y_test_true = result['y_true']
y_test_pred = result['y_pred']

print(f"Model trained successfully!")

In [ ]:
# Compute all metrics
def compute_metrics(y_true, y_pred):
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    bias = float(np.mean(y_pred - y_true))
    
    # Safe MAPE
    mask = np.abs(y_true) > 1.0
    mape = float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100) if mask.sum() > 0 else None
    
    # sMAPE
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom_safe = np.where(denom > 0, denom, 1.0)
    smape = float(np.mean(np.abs(y_true - y_pred) / denom_safe) * 100)
    
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'Bias': bias, 'MAPE': mape, 'sMAPE': smape}

# Train metrics
train_forecast = model.predict(df_train[['ds'] + REGRESSORS])
y_train_pred = np.clip(train_forecast['yhat'].values, 0, None)
y_train_true = df_train['y'].values

train_metrics = compute_metrics(y_train_true, y_train_pred)
test_metrics = compute_metrics(y_test_true, y_test_pred)

print("\n" + "="*60)
print("RESULTATS FINAUX")
print("="*60)
print(f"\nTrain: MAE={train_metrics['MAE']:.3f}, RMSE={train_metrics['RMSE']:.3f}, R2={train_metrics['R2']:.4f}")
print(f"Test:  MAE={test_metrics['MAE']:.3f}, RMSE={test_metrics['RMSE']:.3f}, R2={test_metrics['R2']:.4f}")

## 5. Visualisation

In [ ]:
# Plot forecast
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Full forecast
ax = axes[0, 0]
full_forecast = model.predict(df_agg[['ds'] + REGRESSORS])
ax.plot(df_agg['ds'], df_agg['y'], 'b-', label='Actual', alpha=0.7)
ax.plot(full_forecast['ds'], np.clip(full_forecast['yhat'], 0, None), 'r--', label='Predicted', alpha=0.7)
ax.axvline(x=df_train['ds'].max(), color='g', linestyle=':', label='Train/Test split')
ax.fill_between(full_forecast['ds'], 
                np.clip(full_forecast['yhat_lower'], 0, None), 
                full_forecast['yhat_upper'], 
                alpha=0.2, color='red')
ax.set_title('Prophet Forecast')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Test period zoom
ax = axes[0, 1]
ax.plot(df_test['ds'], y_test_true, 'bo-', label='Actual', markersize=4)
ax.plot(df_test['ds'], y_test_pred, 'rs-', label='Predicted', markersize=4)
ax.set_title(f'Test Period (R2={test_metrics["R2"]:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Scatter plot
ax = axes[1, 0]
ax.scatter(y_test_true, y_test_pred, alpha=0.6)
max_val = max(y_test_true.max(), y_test_pred.max())
ax.plot([0, max_val], [0, max_val], 'r--', label='Perfect')
ax.set_xlabel('Actual')
ax.set_ylabel('Predicted')
ax.set_title('Predicted vs Actual')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Residuals
ax = axes[1, 1]
residuals = y_test_true - y_test_pred
ax.hist(residuals, bins=30, edgecolor='black', alpha=0.7)
ax.axvline(x=0, color='r', linestyle='--')
ax.set_xlabel('Residual')
ax.set_ylabel('Frequency')
ax.set_title(f'Residuals (Bias={test_metrics["Bias"]:.2f})')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ARTIFACTS_PATH / f"prophet_forecast_{TODAY}.png", dpi=150)
plt.show()

In [ ]:
# Prophet components plot
fig = model.plot_components(full_forecast)
plt.tight_layout()
plt.savefig(ARTIFACTS_PATH / f"prophet_components_{TODAY}.png", dpi=150)
plt.show()

## 6. Sauvegarde des Artefacts

In [ ]:
# Save model
model_path = ARTIFACTS_PATH / f"prophet_model_{TODAY}.pkl"
joblib.dump(model, model_path)
print(f"Model saved: {model_path}")

# Save config
config = {
    'best_params': best_params,
    'regressors': REGRESSORS,
    'horizon': HORIZON,
    'optuna_trials': OPTUNA_TRIALS if USE_OPTUNA else None,
    'date_col': DATE_COL,
    'target_col': TARGET_COL,
    'train_dates': len(df_train),
    'test_dates': len(df_test),
}
config_path = ARTIFACTS_PATH / f"prophet_config_{TODAY}.json"
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2, default=str)
print(f"Config saved: {config_path}")

# Save metrics
metrics = {
    'train': train_metrics,
    'test': test_metrics,
    'model_type': 'prophet',
    'timestamp': TODAY,
}
metrics_path = METRICS_PATH / f"prophet_metrics_{TODAY}.json"
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved: {metrics_path}")

In [ ]:
# Summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Model: Prophet (Optimized)")
print(f"Training samples: {len(df_train)}")
print(f"Test samples: {len(df_test)}")
print(f"")
print(f"Test Metrics:")
print(f"  MAE:   {test_metrics['MAE']:.3f}")
print(f"  RMSE:  {test_metrics['RMSE']:.3f}")
print(f"  R2:    {test_metrics['R2']:.4f}")
print(f"  sMAPE: {test_metrics['sMAPE']:.2f}%")
print(f"")
print(f"Artifacts saved to: {ARTIFACTS_PATH}")
print(f"Metrics saved to: {METRICS_PATH}")